# 01 — Setup and Connect to Azure AI Foundry

This notebook guides you through connecting to an Azure AI Foundry project
using the **Microsoft Agent Framework**. It verifies your credentials, endpoint,
and model deployment are working correctly.

## Configuration

Set your agent name below. Change this to work with any registered agent.

In [ ]:
# Change this to any registered agent name
AGENT_NAME = "code-helper"

## Install Dependencies

In [ ]:
!uv pip install --system -e ".." --quiet

## Load Environment

Loads `AZURE_AI_PROJECT_ENDPOINT` from your `.env` file (in the repo root).
Make sure you've copied `.env.example` to `.env` and filled in your endpoint.

---

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")
AZURE_AI_PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
print(f"✓ Endpoint loaded: {AZURE_AI_PROJECT_ENDPOINT[:50]}…")

## Verify Azure Credentials

Tests that `DefaultAzureCredential` can authenticate.
Run `az login` first if you haven't already.

In [ ]:
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default")
print(f"✓ Credential authenticated successfully")
print(f"  Token expires: {token.expires_on}")

## Create Chat Client

Creates an `AzureOpenAIResponsesClient` — the same client the agent factory uses.
This verifies the endpoint and model deployment are reachable.

In [ ]:
from agents.registry import REGISTRY
from agents._base.client import get_chat_client

# Look up the agent's config to get the deployment name
entry = REGISTRY.get_agent(AGENT_NAME)
config = entry.config_class()

print(f"Agent:      {config.agent_name}")
print(f"Deployment: {config.agent_deployment_name}")
print(f"Endpoint:   {config.azure_ai_project_endpoint}")

client = get_chat_client(
    endpoint=config.azure_ai_project_endpoint,
    deployment_name=config.agent_deployment_name,
)
print(f"\n✓ Chat client created: {type(client).__name__}")

## Assemble a Test Agent

Uses the agent factory to create an agent in-process — the same path
as `run_agent.py` and `app.py`.

In [ ]:
from agents._base.agent_factory import create_agent

agent = create_agent(config)
print(f"✓ Agent assembled: {config.agent_name}")
print(f"  Type: {type(agent).__name__}")

## Quick Smoke Test

Send a simple prompt to verify the full pipeline works end-to-end:
environment → credentials → client → agent → model → response.

In [ ]:
result = await agent.run("Say 'Connection OK' and nothing else.")
response_text = result.text if hasattr(result, "text") else str(result)
print(f"✓ Agent response: {response_text}")

## List All Registered Agents

Shows every agent in the registry — these are all available for use.

In [ ]:
print("Registered agents:")
for entry in REGISTRY.list_agents():
    marker = " ← current" if entry.name == AGENT_NAME else ""
    print(f"  • {entry.name} ({entry.config_class.__name__}){marker}")

## Endpoint Diagnostics

Parse the endpoint URL to show project details for troubleshooting.

In [ ]:
endpoint = config.azure_ai_project_endpoint.rstrip("/")
parts = endpoint.split("/api/projects/")
if len(parts) == 2:
    hub_url = parts[0]
    project_name = parts[1]
    print(f"Hub endpoint:  {hub_url}")
    print(f"Project name:  {project_name}")
    print(f"\n→ In Azure AI Foundry portal, verify project '{project_name}'")
    print(f"  has a deployment named '{config.agent_deployment_name}'")
else:
    print(f"Endpoint: {endpoint}")

## Next Steps

- Continue to **02_build_and_run_agent.ipynb** to have a full conversation with an agent
- See **03_scaffold_agent.ipynb** to create a brand new agent